In [2]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [3]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [4]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-20 18:37:35, wtch_dt_end:2026-07-21 18:37:35


In [5]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [6]:
@file:DependsOn("org.json:json:20250107")

In [7]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [8]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2142,1082,0,0.260000,61,5.144163,2.413744,0.250000,3.931417,5.500000,6.410000,19.621000
rtmWqChpla,Comparable<*>,2142,1314,0,,178,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2142,1,0,,2142,null,null,,,,,
rtmWqWtchStaCd,String,2142,14,0,SEA5001,179,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2142,2142,0,1,1,1071.500000,618.486459,1,535.916667,1071.500000,1607.083333,2142
rtmWqTu,Int,2142,160,0,5,197,25.504202,33.740274,0,5.000000,11.000000,32.083333,232
ph,Double,2142,140,0,7.500000,53,7.674790,0.292043,7.020000,7.470000,7.630000,7.920000,9.050000
rtmWqSlnty,Number,2142,1986,0,32.705002,4,21.836755,10.278936,0.020000,13.959000,26.649000,29.555000,34.032001
rtmWqCndctv,Float,2142,2056,0,44.272999,3,34.097408,15.379461,0.046000,23.219249,39.829000,45.028417,54.451000
rtmWqWtchDtlDt,String,2142,190,0,2026-07-20 18:55:00.0,14,null,null,2026-07-20 18:40:00.0,2026-07-21 00:40:00.0,2026-07-21 06:30:00.0,2026-07-21 12:25:00.0,2026-07-21 18:15:00.0


In [16]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0.0 else value.toDouble()
}.convert {  num and rtmWqTu and rtmWqSlnty }.with { it.toString().trim().toDouble() }


df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2142,1082,0,0.260000,61,5.144163,2.413744,0.250000,3.931417,5.500000,6.410000,19.621000
rtmWqChpla,Double,2142,1314,0,0.000000,178,5.260620,5.286769,0.000000,1.301833,3.054000,7.920000,30.180000
rtmWqWtchStaCd,String,2142,14,0,SEA5001,179,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Double,2142,2142,0,1.000000,1,1071.500000,618.486459,1.000000,535.916667,1071.500000,1607.083333,2142.000000
rtmWqTu,Double,2142,160,0,5.000000,197,25.504202,33.740274,0.000000,5.000000,11.000000,32.083333,232.000000
ph,Double,2142,140,0,7.500000,53,7.674790,0.292043,7.020000,7.470000,7.630000,7.920000,9.050000
rtmWqSlnty,Double,2142,1986,0,32.705002,4,21.836755,10.278936,0.020000,13.957916,26.649500,29.555167,34.032001
rtmWqCndctv,Float,2142,2056,0,44.272999,3,34.097408,15.379461,0.046000,23.219249,39.829000,45.028417,54.451000
rtmWqWtchDtlDt,LocalDateTime,2142,190,0,2026-07-20T18:55,14,null,null,2026-07-20T18:40,2026-07-21T00:40,2026-07-21T06:30,2026-07-21T12:25,2026-07-21T18:15
rtmWtchWtem,Double,2142,753,0,29.120001,13,26.379902,2.334195,19.740000,24.980000,26.459999,28.190001,30.660000


In [11]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [17]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1.000000,2026-07-20T18:40,0.950000,2.380000,SEA5002,57.000000,7.390000,16.507000,27.089001,30.219999
2.000000,2026-07-20T18:40,8.170000,30.180000,SEA1002,6.000000,7.630000,26.483000,42.237000,26.070000
3.000000,2026-07-20T18:40,5.046000,0.975000,NEP3001,10.000000,7.850000,24.445000,38.474998,26.280001
4.000000,2026-07-20T18:40,7.780000,2.650000,NEP1002,24.000000,8.070000,1.140000,2.230000,29.200001
5.000000,2026-07-20T18:40,6.380000,2.070000,NEP2002,9.000000,7.980000,28.569000,43.494999,24.110001


In [18]:
removedDf
    .select{  일시 and 수온 and 관측정점코드   }
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온 °C"}
        line{
            color(관측정점코드){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점코드"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="MdXNFS" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("MdXNFS");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"일시":[1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E1